# Running a campaign on a Colab GPU

Colab is one way to get a GPU, not a mode the code is written around. The package is
the same one that runs on a local machine; this notebook only clones it, checks the
runtime, runs a target, and packs the output.

Attach a GPU first: **Runtime > Change runtime type > T4 GPU**.

The training campaigns of appendix O take about eleven T4-hours in total, which is
longer than one Colab session. Run them a group at a time and save after each, or
mount Drive so a recycled runtime does not lose the output.

In [ ]:
REPO = "https://github.com/<user>/EDMGrokking.git"  # set this

import os, pathlib
if not pathlib.Path("EDMGrokking").exists():
    !git clone -q $REPO
%cd EDMGrokking/code
!pip install -q -r requirements-train.txt

In [ ]:
from colab.bootstrap import setup, plan, run, save, mount_drive

setup()   # reports the GPU, pins BLAS threads, fails early if no GPU is attached

## What is there to run

`plan` orders the work and totals its cost before anything starts. `train` selects
every training campaign; a narrower prefix such as `train.perceptron` or a single id
such as `train.perceptron.eos` selects less.

In [ ]:
!python -m actdim list
plan("train")

In [ ]:
# A short one first, to check the whole path works before spending a session on it.
run("train.perceptron.eos", extra=["--fast"])

In [ ]:
run("train.perceptron")

## Getting the results back

`save()` packs `runs/` and offers it as a download. Pass `only=` to pack one group,
since the trajectory sketches run to hundreds of megabytes. Pass `drive_dir=` to copy
the archive to a mounted Drive instead, which survives the runtime being recycled.

On the machine that holds the repository, unpack it into `code/`:

```bash
python -c "import sys; sys.path.insert(0,'.'); from colab.bootstrap import unpack; unpack('~/Downloads/actdim_runs.tar.gz')"
python -m actdim promote train
```

In [ ]:
# mount_drive()                      # optional, for campaigns longer than a session
save(only=["train.perceptron"])    # or save(drive_dir="/content/drive/MyDrive/actdim")